# SC4001 Startup (run these 3 cells top to bottom every session)

BEFORE FIRST USE (once per person)
- Add a shortcut to the shared "SC4001 Project" folder in My Drive and rename it exactly: sc4001_project

EVERY SESSION
1. Run cell 1: mounts Drive, pulls the latest code
2. Run cell 2 & 2a: unpacks the data from Drive into Colab's temporary disk
   (Colab wipes /content when the session disconnects. Drive is the permanent copy.)
3. Run cell 3: checks everything is in place. It should print 5500 twice (original, then cleaned) and then shows splits/splits.csv.

RULES
- Always git pull before working. Never edit splits.csv by hand. Never rename files in data/.
- Save outputs (features, models, results) to Drive, not just /content.
- Reproducibility settings (seed, sizes, CLIP model) live in config.yaml only.
- splits.csv comes from git. Never regenerate with dedupe_and_split.py.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
if os.path.isdir('/content/repo'):
    !git -C /content/repo pull
else:
    !git clone https://github.com/ster2345/SC4001-Grp-Proj.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

In [ ]:
if not os.path.isdir('/content/data/original'):
    !mkdir -p /content/data
    !tar -xf /content/drive/MyDrive/sc4001_project/raw_downloads/original_subset.tar -C /content/data
else:
    print("data already unpacked")

In [ ]:
if not os.path.isdir('/content/data/cleaned'):
    !tar -xf /content/drive/MyDrive/sc4001_project/raw_downloads/cleaned_subset.tar -C /content/data
else:
    print("cleaned data already unpacked")

In [ ]:
!find /content/data/original -type f | wc -l   # expect 5500
!find /content/data/cleaned -type f | wc -l    # expect 5500
!ls splits/splits.csv

In [ ]:
import pandas as pd
pd.set_option("display.width", 200, "display.max_colwidth", 80)
print(pd.read_csv("/content/drive/MyDrive/sc4001_project/raw_downloads/stats_original.csv").to_string())

In [ ]:
import pandas as pd
pd.set_option("display.width", 200, "display.max_colwidth", 80)
print(pd.read_csv("/content/drive/MyDrive/sc4001_project/raw_downloads/stats_cleaned.csv").to_string())

In [ ]:
# find predictions
from pathlib import Path

pred_dir = Path(
    "/content/drive/MyDrive/sc4001_project/results/predictions"
)

files = sorted(pred_dir.glob("*.csv"))

print(f"Found {len(files)} prediction files")

for file in files:
    print(file.name)

In [ ]:
# Open Prediction File
import pandas as pd

file_path = pred_dir / "clip_probe_original.csv"

df = pd.read_csv(file_path)



## Evaluation


In [ ]:
# CLIP (Original)
from sklearn.metrics import accuracy_score, roc_auc_score

test_df = df[df["split"] == "test"].copy()

for generator in ["adm", "biggan", "midjourney"]:
    subset = test_df[test_df["source"] == generator]

    y_true = subset["label"]
    y_score = subset["score"]
    y_pred = (y_score >= 0.5).astype(int)

    accuracy = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_score)

    print(f"\nGenerator: {generator}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"AUC: {auc:.4f}")

In [ ]:
# CLIP (Cleaned)
df_cleaned = pd.read_csv(pred_dir / "clip_probe_cleaned.csv")
test_cleaned = df_cleaned[df_cleaned["split"] == "test"].copy()

for generator in ["adm", "biggan", "midjourney"]:
    subset = test_cleaned[test_cleaned["source"] == generator]

    y_true = subset["label"]
    y_score = subset["score"]
    y_pred = (y_score >= 0.5).astype(int)

    accuracy = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_score)

    print(f"\nGenerator: {generator}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"AUC: {auc:.4f}")

In [ ]:
# ResNet-50 (Original)
for seed in [42, 43, 44]:
    file_path = pred_dir / f"resnet50_original_seed{seed}.csv"
    df_resnet = pd.read_csv(file_path)
    test_resnet = df_resnet[df_resnet["split"] == "test"]

    print(f"\n===== Seed {seed} =====")

    for generator in ["adm", "biggan", "midjourney"]:
        subset = test_resnet[test_resnet["source"] == generator]

        y_true = subset["label"]
        y_score = subset["score"]
        y_pred = (y_score >= 0.5).astype(int)

        accuracy = accuracy_score(y_true, y_pred)
        auc = roc_auc_score(y_true, y_score)

        print(
            f"{generator}: "
            f"Accuracy={accuracy:.4f}, "
            f"AUC={auc:.4f}"
        )

In [ ]:
# ResNet-50 (Cleaned)
for seed in [42, 43, 44]:
    file_path = pred_dir / f"resnet50_cleaned_seed{seed}.csv"
    df_resnet = pd.read_csv(file_path)
    test_resnet = df_resnet[df_resnet["split"] == "test"]

    print(f"\n===== Seed {seed} =====")

    for generator in ["adm", "biggan", "midjourney"]:
        subset = test_resnet[test_resnet["source"] == generator]

        y_true = subset["label"]
        y_score = subset["score"]
        y_pred = (y_score >= 0.5).astype(int)

        accuracy = accuracy_score(y_true, y_pred)
        auc = roc_auc_score(y_true, y_score)

        print(
            f"{generator}: "
            f"Accuracy={accuracy:.4f}, "
            f"AUC={auc:.4f}"
        )

In [ ]:
# Model Comparison
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score

results = []

for model in ["clip_probe", "resnet50"]:
    for version in ["original", "cleaned"]:

        seeds = [None] if model == "clip_probe" else [42, 43, 44]

        for seed in seeds:
            if seed is None:
                filename = f"{model}_{version}.csv"
            else:
                filename = f"{model}_{version}_seed{seed}.csv"

            df_run = pd.read_csv(pred_dir / filename)
            df_test = df_run[df_run["split"] == "test"]

            for generator in ["adm", "biggan", "midjourney"]:
                subset = df_test[df_test["source"] == generator]

                assert set(subset["label"]) == {0, 1}
                assert len(subset) == 1000

                y_true = subset["label"]
                y_score = subset["score"]
                y_pred = (y_score >= 0.5).astype(int)

                results.append({
                    "Model": model,
                    "Training Data": version,
                    "Generator": generator,
                    "Seed": seed,
                    "Accuracy": accuracy_score(y_true, y_pred),
                    "AUC": roc_auc_score(y_true, y_score)
                })

results_df = pd.DataFrame(results)

summary = (
    results_df
    .groupby(["Model", "Training Data", "Generator"])
    .agg(
        Accuracy=("Accuracy", "mean"),
        AUC=("AUC", "mean"),
        Runs=("Accuracy", "count")
    )
    .reset_index()
)

display(summary.round(4))

## Robustness Testing

In [ ]:
# Generate JPEG-Compressed Test Images
from pathlib import Path
from PIL import Image
import pandas as pd

splits = pd.read_csv("/content/repo/splits/splits.csv")

test_images = splits[splits["split"] == "test"].copy()

data_root = Path("/content/data")
output_root = Path("/content/robustness/jpeg")

qualities = [95, 80, 60]

for quality in qualities:
    output_dir = output_root / f"q{quality}"
    output_dir.mkdir(parents=True, exist_ok=True)

    for _, row in test_images.iterrows():
        input_path = data_root / row["path_original"]
        output_path = output_dir / f"{row['image_id']}.jpg"

        if output_path.exists():
            continue

        with Image.open(input_path) as img:
            img = img.convert("RGB")
            img.save(output_path, format="JPEG", quality=quality)

    count = len(list(output_dir.glob("*.jpg")))
    print(f"JPEG quality {quality}: {count} images")

    assert count == len(test_images), (
        f"Expected {len(test_images)}, found {count}"
    )

print("\nJPEG robustness datasets prepared successfully")

In [ ]:
#Generate Resized Test Images
from pathlib import Path
from PIL import Image

data_root = Path("/content/data")
output_root = Path("/content/robustness/resize")

sizes = [128, 64]

for size in sizes:
    output_dir = output_root / f"{size}x{size}"
    output_dir.mkdir(parents=True, exist_ok=True)

    for _, row in test_images.iterrows():
        input_path = data_root / row["path_original"]
        output_path = output_dir / f"{row['image_id']}.png"

        if output_path.exists():
            continue

        with Image.open(input_path) as img:
            img = img.convert("RGB")

            img = img.resize((size, size), Image.Resampling.LANCZOS)
            img = img.resize((256, 256), Image.Resampling.LANCZOS)

            img.save(output_path, format="PNG")

    count = len(list(output_dir.glob("*.png")))

    print(f"Resize {size}x{size}: {count} images")

    assert count == len(test_images)

print("\nResizing robustness datasets prepared successfully")


In [ ]:
# Load ResNet-50 Model
import torch
from torch import nn
from torchvision import models, transforms
from PIL import Image
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

checkpoint_path = Path("/content/drive/MyDrive/sc4001_project/results/checkpoints/resnet50_original_seed42.pt")
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 1)
model.load_state_dict(checkpoint["state_dict"])
model = model.to(device)
model.eval()

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

In [ ]:
# ResNet-50 Original (All Degradations)
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd

conditions = {
    "jpeg95": ("/content/robustness/jpeg/q95", ".jpg"),
    "jpeg80": ("/content/robustness/jpeg/q80", ".jpg"),
    "jpeg60": ("/content/robustness/jpeg/q60", ".jpg"),
    "resize128": ("/content/robustness/resize/128x128", ".png"),
    "resize64": ("/content/robustness/resize/64x64", ".png")
}

class RobustnessDataset(Dataset):
    def __init__(self, test_df, image_dir, extension, preprocess):
        self.test_df = test_df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.extension = extension
        self.preprocess = preprocess

    def __len__(self):
        return len(self.test_df)

    def __getitem__(self, index):
        row = self.test_df.iloc[index]
        path = self.image_dir / (f"{row['image_id']}{self.extension}")

        with Image.open(path) as img:
            image = self.preprocess(img.convert("RGB"))

        return image, row["image_id"], int(row["label"]), row["source"]

output_dir = Path("/content/robustness/resnet_predictions")
output_dir.mkdir(parents=True, exist_ok=True)

all_results = []

for condition, (image_dir, extension) in conditions.items():
    prediction_file = (output_dir / f"resnet50_original_seed42_{condition}.csv")

    if prediction_file.exists():
        print(f"Reusing {condition} predictions")
        predictions = pd.read_csv(prediction_file)
    else:
        print(f"Running inference: {condition}")

        dataset = RobustnessDataset(test_images, image_dir, extension, preprocess)
        loader = DataLoader(dataset, batch_size=32, shuffle=False)

        records = []
        model.eval()

        with torch.no_grad():
            for images, image_ids, labels, sources in loader:
                images = images.to(device)
                scores = torch.sigmoid(model(images)).flatten().cpu().numpy()

                for image_id, label, source, score in zip(image_ids, labels, sources, scores):
                    records.append({
                        "image_id": image_id,
                        "label": int(label),
                        "source": source,
                        "score": float(score)
                    })

        predictions = pd.DataFrame(records)
        predictions.to_csv(prediction_file, index=False)

    for generator in ["adm", "biggan", "midjourney"]:
        subset = predictions[predictions["source"] == generator]

        y_true = subset["label"]
        y_score = subset["score"]

        all_results.append({
            "condition": condition,
            "generator": generator,
            "accuracy": accuracy_score(
                y_true, (y_score >= 0.5).astype(int)
            ),
            "auc": roc_auc_score(y_true, y_score)
        })

robustness_summary = pd.DataFrame(all_results)
display(robustness_summary.round(4))


In [ ]:
# Load ResNet-50 Cleaned Model
cleaned_checkpoint_path = Path(
    "/content/drive/MyDrive/sc4001_project/results/checkpoints/"
    "resnet50_cleaned_seed42.pt"
)

cleaned_checkpoint = torch.load(
    cleaned_checkpoint_path,
    map_location="cpu",
    weights_only=True
)

cleaned_model = models.resnet50(weights=None)
cleaned_model.fc = nn.Linear(cleaned_model.fc.in_features, 1)
cleaned_model.load_state_dict(cleaned_checkpoint["state_dict"])
cleaned_model = cleaned_model.to(device)
cleaned_model.eval()

print("Loaded:", cleaned_checkpoint_path.name)
print("Data version:", cleaned_checkpoint["version"])
print("Seed:", cleaned_checkpoint["seed"])
print("Best epoch:", cleaned_checkpoint["epoch"])


In [ ]:
# ResNet-50 Cleaned (All Degradations)
cleaned_results = []

for condition, (image_dir, extension) in conditions.items():
    prediction_file = (output_dir / f"resnet50_cleaned_seed42_{condition}.csv")

    if prediction_file.exists():
        print(f"Reusing cleaned {condition} predictions")
        predictions = pd.read_csv(prediction_file)
    else:
        print(f"Running cleaned ResNet inference: {condition}")

        dataset = RobustnessDataset(test_images, image_dir, extension, preprocess)
        loader = DataLoader(dataset, batch_size=32, shuffle=False)

        records = []
        cleaned_model.eval()

        with torch.no_grad():
            for images, image_ids, labels, sources in loader:
                images = images.to(device)

                scores = torch.sigmoid(cleaned_model(images)).flatten().cpu().numpy()

                for image_id, label, source, score in zip(image_ids, labels, sources, scores):
                    records.append({
                        "image_id": image_id,
                        "label": int(label),
                        "source": source,
                        "score": float(score)
                    })

        predictions = pd.DataFrame(records)
        predictions.to_csv(prediction_file, index=False)

    for generator in ["adm", "biggan", "midjourney"]:
        subset = predictions[predictions["source"] == generator]

        y_true = subset["label"]
        y_score = subset["score"]

        cleaned_results.append({
            "condition": condition,
            "generator": generator,
            "accuracy": accuracy_score(
                y_true, (y_score >= 0.5).astype(int)
            ),
            "auc": roc_auc_score(y_true, y_score)
        })

cleaned_robustness_summary = pd.DataFrame(cleaned_results)
display(cleaned_robustness_summary.round(4))

In [ ]:
# ResNet-50 Robustness Comparison (Original vs Cleaned)
original_results = robustness_summary.rename(columns={
    "accuracy": "original_accuracy",
    "auc": "original_auc"
})

cleaned_results = cleaned_robustness_summary.rename(columns={
    "accuracy": "cleaned_accuracy",
    "auc": "cleaned_auc"
})

comparison_df = pd.merge(
    original_results,
    cleaned_results,
    on=["condition", "generator"],
    validate="one_to_one"
)

comparison_df["accuracy_change"] = (
    comparison_df["cleaned_accuracy"]
    - comparison_df["original_accuracy"]
)

comparison_df["auc_change"] = (
    comparison_df["cleaned_auc"]
    - comparison_df["original_auc"]
)

assert len(comparison_df) == 15
display(comparison_df.round(4))

In [ ]:
# ResNet-50 Robustness - Additional Seeds (43 and 44)
from pathlib import Path
import torch
import pandas as pd
from torchvision import models
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, roc_auc_score

checkpoint_dir = Path(
    "/content/drive/MyDrive/sc4001_project/results/checkpoints"
)

extra_results = []

for version in ["original", "cleaned"]:
    for seed in [43, 44]:
        checkpoint_path = (checkpoint_dir / f"resnet50_{version}_seed{seed}.pt")

        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True
        )

        extra_model = models.resnet50(weights=None)
        extra_model.fc = nn.Linear(extra_model.fc.in_features, 1)
        extra_model.load_state_dict(checkpoint["state_dict"])
        extra_model = extra_model.to(device).eval()

        print(f"\nLoaded ResNet {version}, seed {seed}")

        for condition, (image_dir, extension) in conditions.items():
            prediction_file = (
                output_dir /
                f"resnet50_{version}_seed{seed}_{condition}.csv"
            )

            if prediction_file.exists():
                predictions = pd.read_csv(prediction_file)
            else:
                dataset = RobustnessDataset(test_images, image_dir, extension, preprocess)
                loader = DataLoader(dataset, batch_size=32, shuffle=False)

                records = []

                with torch.no_grad():
                    for images, image_ids, labels, sources in loader:
                        images = images.to(device)

                        scores = torch.sigmoid(extra_model(images)).flatten().cpu().numpy()

                        for image_id, label, source, score in zip(image_ids, labels, sources, scores):
                            records.append({
                                "image_id": image_id,
                                "label": int(label),
                                "source": source,
                                "score": float(score)
                            })

                predictions = pd.DataFrame(records)
                predictions.to_csv(prediction_file, index=False)

            for generator in ["adm", "biggan", "midjourney"]:
                subset = predictions[predictions["source"] == generator]

                y_true = subset["label"]
                y_score = subset["score"]

                extra_results.append({
                    "version": version,
                    "seed": seed,
                    "condition": condition,
                    "generator": generator,
                    "accuracy": accuracy_score(
                        y_true, (y_score >= 0.5).astype(int)
                    ),
                    "auc": roc_auc_score(y_true, y_score)
                })

        del extra_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

extra_results_df = pd.DataFrame(extra_results)

print("\nAdditional evaluation results:")
display(extra_results_df.round(4))

In [ ]:
# ResNet-50 Robustness - Three Seed Summary
import pandas as pd

seed42_original = robustness_summary.copy()
seed42_original["version"] = "original"
seed42_original["seed"] = 42

seed42_cleaned = cleaned_robustness_summary.copy()
seed42_cleaned["version"] = "cleaned"
seed42_cleaned["seed"] = 42

all_resnet_seeds = pd.concat(
    [
        seed42_original,
        seed42_cleaned,
        extra_results_df
    ],
    ignore_index=True
)

resnet_seed_summary = (
    all_resnet_seeds
    .groupby(["version", "condition", "generator"])
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        n_seeds=("seed", "nunique")
    )
    .reset_index()
)

assert (resnet_seed_summary["n_seeds"] == 3).all(), \
    "Some groups are missing one or more seeds!"

print("Total individual results:", len(all_resnet_seeds))
print("Total averaged results:", len(resnet_seed_summary))

display(resnet_seed_summary.round(4))

In [ ]:
# BACKUP - Multi-Seed ResNet Results
from pathlib import Path
import shutil

results_dir = Path(
    "/content/drive/MyDrive/sc4001_project/results"
)

metrics_dir = results_dir / "metrics"
resnet_destination = (
    results_dir / "predictions/robustness/resnet"
)

metrics_dir.mkdir(parents=True, exist_ok=True)
resnet_destination.mkdir(parents=True, exist_ok=True)

# Save individual seed results
all_resnet_seeds.to_csv(
    metrics_dir / "resnet_all_seeds.csv",
    index=False
)

# Save mean and standard deviation
resnet_seed_summary.to_csv(
    metrics_dir / "resnet_seed_summary.csv",
    index=False
)

# Copy ResNet robustness predictions
shutil.copytree(
    "/content/robustness/resnet_predictions",
    resnet_destination,
    dirs_exist_ok=True
)

print("Multi-seed ResNet results saved!")
print("Metrics:", metrics_dir)
print("Predictions:", resnet_destination)

In [ ]:
# CLIP — Recreate Logistic Regression Classifiers
import numpy as np
import pandas as pd
import yaml
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

with open("/content/repo/config.yaml") as f:
    cfg = yaml.safe_load(f)

splits = pd.read_csv("/content/repo/splits/splits.csv")

clip_classifiers = {}

for version in ["original", "cleaned"]:
    print(f"\n===== CLIP {version} =====")

    feature_path = Path(
        "/content/drive/MyDrive/sc4001_project/"
        f"features/clip_vitb32/{version}.npz"
    )

    z = np.load(feature_path)

    assert np.array_equal(z["image_ids"], splits["image_id"].values)
    assert np.array_equal(z["labels"], splits["label"].values)

    X = z["feats"].astype(np.float64)
    X = X / np.linalg.norm(X, axis=1, keepdims=True)

    y = splits["label"].values

    train_mask = (splits["split"] == "train").values
    val_mask = (splits["split"] == "val").values

    best_model = None
    best_result = None

    for C in cfg["train"]["clip_probe"]["C_grid"]:
        clf = LogisticRegression(
            C=C,
            max_iter=cfg["train"]["clip_probe"]["max_iter"]
        )
        clf.fit(X[train_mask], y[train_mask])

        val_scores = clf.predict_proba(X[val_mask])[:, 1]
        val_acc = accuracy_score(y[val_mask], val_scores >= 0.5)

        val_loss = log_loss(y[val_mask], np.clip(val_scores, 1e-7, 1 - 1e-7))

        result = (val_acc, -val_loss)

        if best_result is None or result > best_result:
            best_result = result
            best_model = clf
            best_C = C

    clip_classifiers[version] = best_model

    print("Selected C:", best_C)

    saved = pd.read_csv(pred_dir / f"clip_probe_{version}.csv")
    saved = saved[saved["split"] == "val"]

    recreated_scores = best_model.predict_proba(X[val_mask])[:, 1]
    recreated = pd.DataFrame({
        "image_id": splits.loc[val_mask, "image_id"].values,
        "new_score": recreated_scores
    })

    comparison = saved.merge(
        recreated,
        on="image_id",
        validate="one_to_one"
    )

    assert len(comparison) == val_mask.sum()

    max_diff = np.max(np.abs(comparison["score"] - comparison["new_score"]))

    print("Maximum validation score difference:", max_diff)

In [ ]:
# Load CLIP Encoder
import torch
import numpy as np
from PIL import Image
from pathlib import Path
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = cfg["clip"]["model"]

clip_model = CLIPModel.from_pretrained(model_name)
clip_model = clip_model.to(device).eval()
clip_processor = CLIPProcessor.from_pretrained(model_name)

In [ ]:
# CLIP — Evaluate All Degradation Conditions
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
from sklearn.metrics import accuracy_score, roc_auc_score

clip_conditions = conditions

clip_output_dir = Path("/content/robustness/clip_predictions")
clip_output_dir.mkdir(parents=True, exist_ok=True)

test_df = test_images.reset_index(drop=True).copy()
assert len(test_df) == 3000

clip_results = []

for condition, (image_dir, extension) in clip_conditions.items():
    print(f"\nProcessing {condition}")

    feature_file = clip_output_dir / f"{condition}_features.npz"

    if feature_file.exists():
        print("Reusing saved CLIP features")
        with np.load(feature_file, allow_pickle=True) as z:
            X = z["feats"]
            saved_ids = z["image_ids"].astype(str)

        assert np.array_equal(saved_ids, test_df["image_id"].astype(str).to_numpy())
    else:
        feature_batches = []

        image_paths = [
            Path(image_dir) / f"{image_id}{extension}"
            for image_id in test_df["image_id"]
        ]

        assert all(path.exists() for path in image_paths)

        clip_model.eval()

        with torch.no_grad():
            for start in range(0, len(image_paths), 32):
                batch_paths = image_paths[start:start + 32]

                images = []
                for path in batch_paths:
                    with Image.open(path) as img:
                        images.append(img.convert("RGB"))

                inputs = clip_processor(images=images, return_tensors="pt")

                pixels = inputs["pixel_values"].to(device)

                features = clip_model.get_image_features(pixel_values=pixels)

                if not torch.is_tensor(features):
                    features = features.pooler_output

                feature_batches.append(features.float().cpu().numpy())

        X = np.concatenate(feature_batches).astype(np.float64)

        norms = np.linalg.norm(X, axis=1, keepdims=True)
        assert np.all(norms > 0)
        X = X / norms

        np.savez(feature_file, feats=X, image_ids=test_df["image_id"].astype(str).to_numpy())

    assert X.shape == (3000, 512)
    assert np.isfinite(X).all()

    for version in ["original", "cleaned"]:
        scores = clip_classifiers[version].predict_proba(X)[:, 1]

        predictions = test_df[["image_id", "label", "source"]].copy()

        predictions["score"] = scores

        predictions.to_csv(
            clip_output_dir / f"clip_{version}_{condition}.csv",
            index=False
        )

        for generator in ["adm", "biggan", "midjourney"]:
            subset = predictions[predictions["source"] == generator]

            y_true = subset["label"]
            y_score = subset["score"]

            clip_results.append({
                "condition": condition,
                "version": version,
                "generator": generator,
                "accuracy": accuracy_score(
                    y_true, (y_score >= 0.5).astype(int)
                ),
                "auc": roc_auc_score(y_true, y_score)
            })

clip_robustness_summary = pd.DataFrame(clip_results)

assert len(clip_robustness_summary) == 30

display(clip_robustness_summary.round(4))

In [ ]:
# Preliminary CLIP vs ResNet Comparison (Seed 42)
import pandas as pd

resnet_original = robustness_summary.copy()
resnet_original["model"] = "ResNet-50"
resnet_original["version"] = "original"

resnet_cleaned = cleaned_robustness_summary.copy()
resnet_cleaned["model"] = "ResNet-50"
resnet_cleaned["version"] = "cleaned"

clip_results_table = clip_robustness_summary.copy()
clip_results_table["model"] = "CLIP"

all_robustness_results = pd.concat(
    [resnet_original, resnet_cleaned, clip_results_table],
    ignore_index=True
)

assert len(all_robustness_results) == 60
assert not all_robustness_results.duplicated(["model", "version", "condition", "generator"]).any()

preliminary_comparison = all_robustness_results.pivot(
    index=["version", "condition", "generator"],
    columns="model",
    values=["accuracy", "auc"]
)

preliminary_comparison.columns = [
    f"{metric}_{model}"
    for metric, model in preliminary_comparison.columns
]

preliminary_comparison = preliminary_comparison.reset_index()

preliminary_comparison["accuracy_gap"] = (
    preliminary_comparison["accuracy_CLIP"]
    - preliminary_comparison["accuracy_ResNet-50"]
)

preliminary_comparison["auc_gap"] = (
    preliminary_comparison["auc_CLIP"]
    - preliminary_comparison["auc_ResNet-50"]
)

assert len(preliminary_comparison) == 30

display(preliminary_comparison.round(4))

In [ ]:
# CLIP vs ResNet - Updated Multi-Seed Robustness Comparison
import pandas as pd

clip_results = all_robustness_results[
    all_robustness_results["model"] == "CLIP"
][
    ["version", "condition", "generator", "accuracy", "auc"]
].copy()

clip_results = clip_results.rename(columns={
    "accuracy": "clip_accuracy",
    "auc": "clip_auc"
})

resnet_results = resnet_seed_summary[
    [
        "version", "condition", "generator",
        "accuracy_mean", "accuracy_std",
        "auc_mean", "auc_std"
    ]
].copy()

updated_comparison = pd.merge(
    clip_results,
    resnet_results,
    on=["version", "condition", "generator"],
    validate="one_to_one"
)

updated_comparison["accuracy_gap"] = (
    updated_comparison["clip_accuracy"] -
    updated_comparison["accuracy_mean"]
)

updated_comparison["auc_gap"] = (
    updated_comparison["clip_auc"] -
    updated_comparison["auc_mean"]
)

assert len(updated_comparison) == 30

print("Total comparisons:", len(updated_comparison))

display(updated_comparison.round(4))

## Robustness Visualisation


In [ ]:
import matplotlib.pyplot as plt

condition_groups = {
    "JPEG Compression": ["jpeg95", "jpeg80", "jpeg60"],
    "Resizing": ["resize128", "resize64"]
}

for group_name, conditions_list in condition_groups.items():
    for generator in ["adm", "biggan", "midjourney"]:
        plt.figure(figsize=(8, 5))

        for version in ["original", "cleaned"]:
            subset = updated_comparison[
                (updated_comparison["generator"] == generator) &
                (updated_comparison["version"] == version)
            ].set_index("condition").loc[conditions_list]

            plt.plot(
                conditions_list,
                subset["clip_accuracy"] * 100,
                marker="o",
                label=f"CLIP ({version})"
            )

            plt.errorbar(
                conditions_list,
                subset["accuracy_mean"] * 100,
                yerr=subset["accuracy_std"] * 100,
                marker="s",
                capsize=4,
                label=f"ResNet-50 ({version})"
            )

        plt.xlabel("Degradation condition")
        plt.ylabel("Accuracy (%)")
        plt.title(f"{generator.upper()}: {group_name} Robustness")
        plt.ylim(0, 100)
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
# BACKUP - Final Comparison and Figures
from pathlib import Path
import matplotlib.pyplot as plt

backup_dir = Path(
    "/content/drive/MyDrive/sc4001_project/results/metrics"
)
backup_dir.mkdir(parents=True, exist_ok=True)

figures_dir = Path(
    "/content/drive/MyDrive/sc4001_project/results/figures"
)
figures_dir.mkdir(parents=True, exist_ok=True)

updated_comparison.to_csv(
    backup_dir / "updated_clip_vs_resnet_comparison.csv",
    index=False
)

condition_groups = {
    "jpeg": ["jpeg95", "jpeg80", "jpeg60"],
    "resize": ["resize128", "resize64"]
}

for group_name, conditions_list in condition_groups.items():
    for generator in ["adm", "biggan", "midjourney"]:

        fig, ax = plt.subplots(figsize=(8, 5))

        for version in ["original", "cleaned"]:

            subset = updated_comparison[
                (updated_comparison["generator"] == generator) &
                (updated_comparison["version"] == version)
            ].set_index("condition").loc[conditions_list]

            ax.plot(
                conditions_list,
                subset["clip_accuracy"] * 100,
                marker="o",
                label=f"CLIP ({version})"
            )

            ax.errorbar(
                conditions_list,
                subset["accuracy_mean"] * 100,
                yerr=subset["accuracy_std"] * 100,
                marker="s",
                capsize=4,
                label=f"ResNet-50 ({version})"
            )

        ax.set_xlabel("Degradation condition")
        ax.set_ylabel("Accuracy (%)")
        ax.set_title(
            f"{generator.upper()}: {group_name.upper()} Robustness"
        )
        ax.set_ylim(0, 100)
        ax.grid(alpha=0.3)
        ax.legend()
        fig.tight_layout()

        filename = f"{generator}_{group_name}_robustness.png"

        fig.savefig(
            figures_dir / filename,
            dpi=300,
            bbox_inches="tight"
        )

        plt.close(fig)

        print(f"Saved: {filename}")

print("\nSaved comparison table and all six figures!")


## Failure Analysis

In [ ]:
# False Positives and False Negatives
import pandas as pd
from pathlib import Path

resnet_pred = pd.read_csv(
    "/content/robustness/resnet_predictions/"
    "resnet50_original_seed42_jpeg80.csv"
)

clip_pred = pd.read_csv(
    "/content/robustness/clip_predictions/"
    "clip_original_jpeg80.csv"
)

resnet_pred["model"] = "ResNet-50"
clip_pred["model"] = "CLIP"

failure_df = pd.concat(
    [resnet_pred, clip_pred],
    ignore_index=True
)

failure_df["predicted_label"] = (failure_df["score"] >= 0.5).astype(int)

def get_error_type(row):
    if row["label"] == 0 and row["predicted_label"] == 1:
        return "False Positive"
    elif row["label"] == 1 and row["predicted_label"] == 0:
        return "False Negative"
    else:
        return "Correct"

failure_df["error_type"] = failure_df.apply(get_error_type, axis=1)

error_counts = pd.crosstab(
    [failure_df["model"], failure_df["source"]],
    failure_df["error_type"]
)

display(error_counts)

In [ ]:
# Identify Misclassified Images
clip_errors = failure_df[
    failure_df["model"] == "CLIP"
][["image_id", "source", "label", "score", "error_type"]].copy()

resnet_errors = failure_df[
    failure_df["model"] == "ResNet-50"
][["image_id", "source", "label", "score", "error_type"]].copy()

matched = pd.merge(
    clip_errors,
    resnet_errors,
    on=["image_id", "source", "label"],
    suffixes=("_clip", "_resnet"),
    validate="one_to_one"
)

both_wrong = matched[
    (matched["error_type_clip"] != "Correct") &
    (matched["error_type_resnet"] != "Correct")
].copy()

for generator in ["adm", "biggan", "midjourney"]:
    subset = both_wrong[both_wrong["source"] == generator]

    print(f"\n===== {generator.upper()} =====")
    print("Images both models got wrong:", len(subset))

    display(
        subset[[
            "image_id", "label",
            "score_clip", "score_resnet"
        ]].head(5)
    )

In [ ]:
# Visualise Shared Misclassification
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

fig, axes = plt.subplots(3, 2, figsize=(10, 12))

for row_index, generator in enumerate(["adm", "biggan", "midjourney"]):
    samples = both_wrong[both_wrong["source"] == generator].head(2)
    for col_index, (_, sample) in enumerate(samples.iterrows()):
        image_id = sample["image_id"]

        image_path = Path(
            "/content/robustness/jpeg/q80"
        ) / f"{image_id}.jpg"

        with Image.open(image_path) as img:
            axes[row_index, col_index].imshow(img.convert("RGB"))

        axes[row_index, col_index].set_title(
            f"{generator.upper()}\n"
            f"CLIP P(fake): {sample['score_clip']:.3f}\n"
            f"ResNet P(fake): {sample['score_resnet']:.3f}",
            fontsize=10
        )

        axes[row_index, col_index].axis("off")

plt.suptitle(
    "AI-Generated Images Misclassified as Real by Both Models",
    fontsize=14
)

plt.tight_layout()
plt.show()

In [ ]:
# Visualise False Positives
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

false_positives = failure_df[failure_df["error_type"] == "False Positive"].copy()

fig, axes = plt.subplots(2, 2, figsize=(10, 9))

for row_index, model_name in enumerate(["CLIP", "ResNet-50"]):
    samples = (
        false_positives[false_positives["model"] == model_name]
        .sort_values("score", ascending=False)
        .head(2)
    )

    for col_index, (_, sample) in enumerate(samples.iterrows()):
        image_id = sample["image_id"]

        image_path = Path(
            "/content/robustness/jpeg/q80"
        ) / f"{image_id}.jpg"

        with Image.open(image_path) as img:
            axes[row_index, col_index].imshow(img.convert("RGB"))

        axes[row_index, col_index].set_title(
            f"{model_name} — {sample['source'].upper()}\n"
            f"P(fake): {sample['score']:.3f}",
            fontsize=11
        )

        axes[row_index, col_index].axis("off")

plt.suptitle(
    "Real Images Incorrectly Classified as AI-Generated",
    fontsize=14
)

plt.tight_layout()
plt.show()

In [ ]:
# BACKUP - Predictions and Analysis Tables
from pathlib import Path
import shutil

results_dir = Path(
    "/content/drive/MyDrive/sc4001_project/results"
)

metrics_dir = results_dir / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)

summary.to_csv(
    metrics_dir / "main_model_comparison.csv",
    index=False
)

all_robustness_results.to_csv(
    metrics_dir / "all_robustness_results.csv",
    index=False
)

preliminary_comparison.to_csv(
    metrics_dir / "clip_vs_resnet_seed42_comparison.csv",
    index=False
)

error_counts.to_csv(
    metrics_dir / "failure_counts.csv"
)

source_dir = Path("/content/robustness")

destinations = {
    "resnet_predictions": (
        results_dir / "predictions/robustness/resnet"
    ),
    "clip_predictions": (
        results_dir / "predictions/robustness/clip"
    )
}

for folder, destination in destinations.items():
    source = source_dir / folder

    destination.mkdir(parents=True, exist_ok=True)

    shutil.copytree(
        source,
        destination,
        dirs_exist_ok=True
    )

    print(f"Saved: {folder}")

print("\nAll results backed up to shared Drive!")